In [94]:
import pandas as pd

# we are using E commerce churn data to predict whether a user will churn or not
df = pd.read_csv('train_data.csv')

print(df.shape)

print(df.head)

print(df.info)

(5630, 21)
<bound method NDFrame.head of       Unnamed: 0  CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  \
0              0       50001      1     4.0         Mobile Phone         3   
1              1       50002      1     NaN                Phone         1   
2              2       50003      1     NaN                Phone         1   
3              3       50004      1     0.0                Phone         3   
4              4       50005      1     0.0                Phone         1   
...          ...         ...    ...     ...                  ...       ...   
5625        5625       55626      0    10.0             Computer         1   
5626        5626       55627      0    13.0         Mobile Phone         1   
5627        5627       55628      0     1.0         Mobile Phone         1   
5628        5628       55629      0    23.0             Computer         3   
5629        5629       55630      0     8.0         Mobile Phone         1   

      WarehouseToHome 

In [95]:
# out target column is Churn so we will first check the ratio of that in the data
churn_counts = df['Churn'].value_counts(normalize=True) * 100
print("Churn Distribution (%):\n", churn_counts)
# E-commerce datasets are usually imbalanced which is the case as churn is 16.8%.
# Models like Logistic Regression and SVM will need proper evaluation metrics
# rather than relying solely on accuracy.

Churn Distribution (%):
 Churn
0    83.161634
1    16.838366
Name: proportion, dtype: float64


In [96]:
# Data cleaning and categorization

# Drop unique values as not relevant to classification and index data
df = df.drop(df.columns[[0,1]], axis=1)

X = df.drop(columns=['Churn'])
y = df['Churn'] # remove target variable from numerical list

# List down numeric columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.to_list()
cat_cols = X.select_dtypes(include=['object']).columns.to_list()


print('Numerical Features:', num_cols)
print('Categorical Features:', cat_cols)

df[num_cols].describe().T

Numerical Features: ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
Categorical Features: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']


/var/folders/g5/r4tybhdd1nqc3j64spskwhb80000gp/T/ipykernel_57731/1782024449.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.to_list()


,count,mean,std,min,25%,50%,75%,max
Tenure,5366.0,10.189899,8.557241,0.0,2.00,9.00,16.0000,61.00
CityTier,5630.0,1.654707,0.915389,1.0,1.00,1.00,3.0000,3.00
WarehouseToHome,5379.0,15.639896,8.531475,5.0,9.00,14.00,20.0000,127.00
HourSpendOnApp,5375.0,2.931535,0.721926,0.0,2.00,3.00,3.0000,5.00
NumberOfDeviceRegistered,5630.0,3.688988,1.023999,1.0,3.00,4.00,4.0000,6.00
SatisfactionScore,5630.0,3.066785,1.380194,1.0,2.00,3.00,4.0000,5.00
NumberOfAddress,5630.0,4.214032,2.583586,1.0,2.00,3.00,6.0000,22.00
Complain,5630.0,0.284902,0.451408,0.0,0.00,0.00,1.0000,1.00
OrderAmountHikeFromlastYear,5365.0,15.707922,3.675485,11.0,13.00,15.00,18.0000,26.00
CouponUsed,5374.0,1.751023,1.894621,0.0,1.00,1.00,2.0000,16.00


In [97]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# start data preprocessing
# pipeline for numerical type
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline for categorical data
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# combine pipelines into single transformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [98]:
from sklearn.model_selection import train_test_split

# split the data into train and test and preprocess
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\n--- Preprocessing Complete ---")
print(f"Original Training Features Shape: {X_train.shape}")
print(f"Processed Training Features Shape (after One-Hot Encoding): {X_train_processed.shape}")
print(f"Test Set Target Distribution:\n{y_test.value_counts()}")


--- Preprocessing Complete ---
Original Training Features Shape: (4504, 18)
Processed Training Features Shape (after One-Hot Encoding): (4504, 34)
Test Set Target Distribution:
Churn
0    936
1    190
Name: count, dtype: int64


In [99]:
# Model training and evaluation
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=7, random_state=42),
    'K Nearest Neighbor': KNeighborsClassifier(n_neighbors=5),
    'Gaussian Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
}

results = []

for name, model in models.items():
    # Fit model on preprocessed training data
    model.fit(X_train_processed, y_train)

    # Predict label and probabilities on test set
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Calculate performance metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "MCC": mcc,
        "ROC AUC": roc_auc
    })

/Users/shubham.bhendarkar/Documents/bits-sem01/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [100]:
# Create comparison
results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

# Format outputs as percentages for readability
formatted_df = results_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC", "MCC"]:
    formatted_df[col] = formatted_df[col].apply(lambda x: f"{x * 100:.2f}%")

print("=" * 80)
print("                 E-COMMERCE CHURN MODEL PERFORMANCE COMPARISON")
print("=" * 80)
print(formatted_df.to_string(index=False))
print("=" * 80)

                 E-COMMERCE CHURN MODEL PERFORMANCE COMPARISON
                 Model Accuracy Precision Recall F1 Score    MCC ROC AUC
    K Nearest Neighbor   92.27%    87.05% 63.68%   73.56% 70.32%  80.88%
         Decision Tree   90.94%    80.99% 60.53%   69.28% 65.03%  78.82%
Support Vector Machine   91.47%    89.17% 56.32%   69.03% 66.67%  77.46%
   Logistic Regression   88.81%    74.24% 51.58%   60.87% 55.82%  73.97%
  Gaussian Naive Bayes   70.07%    32.20% 70.00%   44.11% 31.15%  70.04%
